# Integration Test Runner

Runs the full test suite for the **server-source-of-truth + batching** implementation (issues #6/#7).

## Structure

| Section | GPU? | What it tests |
|---------|------|----------------|
| 1 — Setup | — | Working dir, GPU check, config |
| 2 — Unit tests | No | ZarrLogger Phase 1, AsyncActivationWriter, ModelAdapter |
| 3 — HF Adapter (GPU) | Yes | Single/batch inference shapes, logprobs, Zarr round-trip |
| 4 — Server + Async Writer (GPU) | Yes | HTTP → Zarr persistence, restart safety |
| 5 — Batch vs Sequential (GPU) | Yes | Per-request activation consistency |
| 6 — Resume from Zarr (GPU) | Yes | Skip already-done prompts, crash recovery, jsonl export |

**Run sections 3-6 on a machine with a GPU (H100 or similar).**

---
## 1 — Setup

In [ ]:
import os
repo_root = "/workspaces/HalluLens"
os.chdir(repo_root)
print(f"cwd: {os.getcwd()}")

In [ ]:
import subprocess, sys

def run_pytest(args: list[str]) -> int:
    """Run pytest with the given args, streaming output to the notebook."""
    cmd = [sys.executable, "-m", "pytest"] + args
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=False)
    return result.returncode

In [ ]:
import torch

GPU_AVAILABLE = torch.cuda.is_available()
if GPU_AVAILABLE:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {props.name}  total={total/1e9:.1f} GB  free={free/1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — sections 3-6 will be skipped.")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
# Model used for all GPU integration tests.
# Must be accessible from HuggingFace Hub or downloaded locally.
TEST_MODEL_NAME = os.environ.get(
    "TEST_MODEL_NAME", "meta-llama/Llama-3.1-8B-Instruct"
)

# Pytest verbosity: "-v" for verbose, "-q" for quiet
VERBOSITY = "-v"

# Set the env var so integration test conftest picks it up
os.environ["TEST_MODEL_NAME"] = TEST_MODEL_NAME

print(f"TEST_MODEL_NAME = {TEST_MODEL_NAME}")

---
## 2 — Unit Tests (no GPU required)

These cover:
- `ZarrActivationsLogger`: `log_metadata`, overwrite support, `get_entry` fix
- `AsyncActivationWriter`: drain, routing, error handling, counters
- `ModelAdapter`: `InferenceResult`, adapter interface, target-layer resolution, activation/logprob extraction with synthetic tensors

In [ ]:
rc = run_pytest([
    "tests/test_zarr_logger_phase1.py",
    "tests/test_async_activation_writer.py",
    "tests/test_model_adapter.py",
    VERBOSITY, "--tb=short",
])
print(f"\nUnit tests exit code: {rc}  ({'PASS' if rc == 0 else 'FAIL'})")

---
## 3 — HFTransformersAdapter GPU Tests

Tests `HFTransformersAdapter` end-to-end on a real model:
- Single-prompt inference: response text, activation shapes, logprob validity
- Target-layer modes: `all`, `first_half`, `second_half`
- Batch vs sequential: response text should match exactly; activations within float16 tolerance
- Zarr round-trip: activations stored and retrieved correctly

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping — no GPU.")
else:
    rc = run_pytest([
        "tests/integration/test_hf_adapter_gpu.py",
        VERBOSITY, "--tb=short", "-m", "not slow",
    ])
    print(f"\nHF Adapter tests exit code: {rc}  ({'PASS' if rc == 0 else 'FAIL'})")

---
## 4 — Server + AsyncActivationWriter GPU Tests

Starts a real FastAPI server process and hits it with HTTP requests:
- Activation entries appear in Zarr after inference
- Entry metadata fields (prompt, response, prompt_hash) are present
- Activation arrays are loadable as tensors
- Server restart does not produce duplicate Zarr rows (overwrite path)

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping — no GPU.")
else:
    rc = run_pytest([
        "tests/integration/test_server_async_gpu.py",
        VERBOSITY, "--tb=short",
    ])
    print(f"\nServer async tests exit code: {rc}  ({'PASS' if rc == 0 else 'FAIL'})")

---
## 5 — Batch Inference Correctness GPU Tests

- `infer_batch([p])` matches `infer(p)` for greedy decoding
- Per-request activations from a 2-prompt batch match sequential results within tolerance
- All batch results stored as separate rows in Zarr (no row clobbering)
- Activation shapes round-trip through Zarr correctly

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping — no GPU.")
else:
    rc = run_pytest([
        "tests/integration/test_batch_inference_gpu.py",
        VERBOSITY, "--tb=short",
    ])
    print(f"\nBatch inference tests exit code: {rc}  ({'PASS' if rc == 0 else 'FAIL'})")

---
## 6 — Resume from Zarr GPU Tests

- Completed prompts are skipped by the resume logic (correct `prompt_hash` matching)
- Full run + identical resume = same row count (no duplicates)
- Graceful shutdown (SIGTERM) drains async writer; entries survive
- Hard kill (SIGKILL) leaves Zarr store readable
- `export_generation_jsonl()` produces correct JSONL matching Zarr responses

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping — no GPU.")
else:
    rc = run_pytest([
        "tests/integration/test_resume_zarr_gpu.py",
        VERBOSITY, "--tb=short",
    ])
    print(f"\nResume tests exit code: {rc}  ({'PASS' if rc == 0 else 'FAIL'})")

---
## Full GPU suite (all integration tests in one shot)

Run all GPU integration tests together — equivalent to running sections 3-6 sequentially.
Use this cell instead of the individual sections when you want a single pass/fail.

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping — no GPU.")
else:
    rc = run_pytest([
        "tests/integration/",
        VERBOSITY, "--tb=short", "-m", "not slow",
    ])
    print(f"\nFull integration suite exit code: {rc}  ({'PASS' if rc == 0 else 'FAIL'})")

---
## Throughput benchmark (slow — opt-in)

Measures batch vs sequential inference throughput on the H100.
Requires `pytest-benchmark`: `pip install pytest-benchmark`.

Expected result: `infer_batch(N)` should be measurably faster than `N × infer(p)` for N ≥ 4.

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping — no GPU.")
else:
    rc = run_pytest([
        "tests/integration/test_hf_adapter_gpu.py",
        "-v", "--tb=short", "-m", "slow", "--benchmark-only",
    ])
    print(f"\nThroughput benchmark exit code: {rc}")